# Agentic RAG: Prerequisite & Course Planning Assistant

**Institution:** Georgia Institute of Technology (Georgia Tech)

**Catalog:** https://catalog.gatech.edu

**Built with:** LangChain · CrewAI · Groq (llama-3.3-70b) · Google Embeddings · FAISS

This system answers student course-planning questions strictly grounded
in the Georgia Tech course catalog. It uses 4 AI agents working in sequence:

1. **Intake Agent** — collects the student profile and asks clarifying questions
2. **Retrieval Agent** — searches the catalog using RAG and returns cited excerpts
3. **Planner Agent** — builds a course plan grounded in catalog evidence
4. **Verifier Agent** — audits every claim for missing citations

## Pipeline
Catalog pages → Chunking → Embeddings → FAISS index → Agent crew → Cited answer

## Run order
Run all cells top to bottom. Do NOT skip any cell.

## Installing Libraries
 - crewai==0.28.8 for LangChain LLM compatibility
 - langchain-groq provides the ChatGroq wrapper
 - faiss-cpu is the vector similarity index
 - langchain-community provides document loaders and FAISS integration.

In [ ]:
# Wipe all conflicting packages completely
!pip uninstall -y langchain langchain-core langchain-groq langchain-community \
    langchain-google-genai crewai crewai-tools 2>/dev/null || true

# Install exact compatible set — tested to work together
!pip install -q "langchain-core==0.3.29"
!pip install -q "langchain==0.3.14"
!pip install -q "langchain-groq==0.2.4"
!pip install -q "langchain-google-genai==2.0.8"
!pip install -q "langchain-community==0.3.14"
!pip install -q "crewai==0.80.0"
!pip install -q faiss-cpu pypdf beautifulsoup4 requests tiktoken

print("Install complete — NOW restart your runtime!")

print("Done! All packages installed.")

Found existing installation: langchain 0.3.14
Uninstalling langchain-0.3.14:
  Successfully uninstalled langchain-0.3.14
Found existing installation: langchain-core 0.3.63
Uninstalling langchain-core-0.3.63:
  Successfully uninstalled langchain-core-0.3.63
Found existing installation: langchain-groq 0.2.4
Uninstalling langchain-groq-0.2.4:
  Successfully uninstalled langchain-groq-0.2.4
Found existing installation: langchain-community 0.3.14
Uninstalling langchain-community-0.3.14:
  Successfully uninstalled langchain-community-0.3.14
Found existing installation: langchain-google-genai 2.0.8
Uninstalling langchain-google-genai-2.0.8:
  Successfully uninstalled langchain-google-genai-2.0.8
Found existing installation: crewai 0.80.0
Uninstalling crewai-0.80.0:
  Successfully uninstalled crewai-0.80.0
Found existing installation: crewai-tools 0.76.0
Uninstalling crewai-tools-0.76.0:
  Successfully uninstalled crewai-tools-0.76.0
ERROR: pip's dependency resolver does not currently take int

## Setting API Key
Used environment variables so no library needs special config.

Groq handles LLM inference;
https://console.groq.com/keys

Google handles text-embedding-004 vector generation.
https://aistudio.google.com/app/api-keys

In [82]:
import os, time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from google.colab import userdata

# Set the Groq API key directly
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Set the Google API key directly

# Use this string directly as llm= in every agent
LLM_MODEL = "groq/llama-3.1-8b-instant"

# Embeddings still need LangChain wrapper
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key= userdata.get('google_api_key') # Set the Google API key directly
)

print("LLM_MODEL set to:", LLM_MODEL)
test_vec = embedding_model.embed_query("test")
print(f"Embedding dim: {len(test_vec)}")

LLM_MODEL set to: groq/llama-3.1-8b-instant
Embedding dim: 3072


## Intializing LLM (Groq) and Embedding model(Google)

ChatGroq implements the same BaseChatModel interface as ChatGoogleGenerativeAI — CrewAI accepts it identically.

llama-3.3-70b outperforms many commercial models on reasoning. text-embedding-004 produces 768-dimensional dense vectors — the standard for semantic search tasks

In [ ]:
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from crewai import LLM
import os

llm = LLM(
    model = "groq/llama-3.3-70b-versatile",
    temperature=0.2
)

embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
)

test = llm.call(
    [{"role": "user", "content": "Say exactly: Groq is working"}]
)
print(test)

vec = embedding_model.embed_query("test")
print(f"Embedding dim: {len(vec)}")

Groq is working
Embedding dim: 3072


## Scraping catalog documents
Used requests + BeautifulSoup for HTML scraping, stripping nav/footer/script tags to extract clean main content.

Each doc stores URL and date_accessed for the required Sources section.

Target: course pages, degree requirements, and academic policy pages to meet the 25+ document minimum.

In [ ]:
import requests, json
from bs4 import BeautifulSoup
from datetime import date

urls = [
  # Course pages
  "https://catalog.gatech.edu/coursesaz/acct/",
  "https://catalog.gatech.edu/coursesaz/ae/",
  "https://catalog.gatech.edu/coursesaz/bios/",
  "https://catalog.gatech.edu/coursesaz/chem/",
  "https://catalog.gatech.edu/coursesaz/cs/",
  "https://catalog.gatech.edu/coursesaz/cse/",
  "https://catalog.gatech.edu/coursesaz/econ/",
  "https://catalog.gatech.edu/coursesaz/ece/",
  "https://catalog.gatech.edu/coursesaz/free/",
  "https://catalog.gatech.edu/coursesaz/gt/",
  "https://catalog.gatech.edu/coursesaz/math/",
  "https://catalog.gatech.edu/coursesaz/me/",
  "https://catalog.gatech.edu/coursesaz/msl/",
  "https://catalog.gatech.edu/coursesaz/phys/",
  "https://catalog.gatech.edu/coursesaz/sci/",
  "https://catalog.gatech.edu/coursesaz/ss/",
  "https://catalog.gatech.edu/coursesaz/soc/",
  "https://catalog.gatech.edu/coursesaz/span/",
  "https://catalog.gatech.edu/coursesaz/russ/",
  "https://catalog.gatech.edu/coursesaz/hist/",
  "https://catalog.gatech.edu/coursesaz/japn/",
  "https://catalog.gatech.edu/coursesaz/engl/",
  "https://catalog.gatech.edu/coursesaz/fs/",
  "https://catalog.gatech.edu/coursesaz/id/",
  "https://catalog.gatech.edu/coursesaz/musi/",
  #degree
  "https://catalog.gatech.edu/rules/13/",
  "https://catalog.gatech.edu/rules/14/",
  #policies
  "https://catalog.gatech.edu/policies/honor-code/",
  "https://catalog.gatech.edu/policies/alcohol-policy/",
  "https://catalog.gatech.edu/policies/ferpa/",
  "https://catalog.gatech.edu/policies/discrimination/",
  "https://catalog.gatech.edu/policies/grading-gpa/scholastic-average/",
  "https://catalog.gatech.edu/policies/grading-gpa/auditing/",
  "https://catalog.gatech.edu/policies/grading-gpa/examination-term-grades/",
  "https://catalog.gatech.edu/policies/grading-gpa/grade-substitution/",
  "https://catalog.gatech.edu/policies/grading-gpa/grading-system/",
  "https://catalog.gatech.edu/policies/grading-gpa/pass-fail-system-rules/",
  "https://catalog.gatech.edu/policies/grading-gpa/progress-reports/",

  # auto-extract from these pages in next step
]
print(f"Ready to scrape {len(urls)} seed URLs")

Ready to scrape 38 seed URLs


In [ ]:
docs = []
headers = {"User-Agent": "Mozilla/5.0"}

for url in urls:
    try:
        r = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["nav","footer","script","style","header"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        if len(text) > 200:   # skip empty pages
            docs.append({"url": url, "text": text, "date": str(date.today())})
            print(f"OK ({len(text):,} chars): {url[:55]}")
        time.sleep(1)
    except Exception as e:
        print(f"FAIL: {url[:55]} — {e}")

with open("catalog_docs.json","w") as f:
    json.dump(docs, f, indent=2)

total_words = sum(len(d["text"].split()) for d in docs)
print(f"\nDone! {len(docs)} docs, {total_words:,} total words")
print("PASS" if total_words > 30000 else "WARNING: need more pages")

OK (846 chars): https://catalog.gatech.edu/coursesaz/acct/
OK (41,106 chars): https://catalog.gatech.edu/coursesaz/ae/
OK (20,684 chars): https://catalog.gatech.edu/coursesaz/bios/
OK (30,242 chars): https://catalog.gatech.edu/coursesaz/chem/
OK (88,251 chars): https://catalog.gatech.edu/coursesaz/cs/
OK (9,412 chars): https://catalog.gatech.edu/coursesaz/cse/
OK (37,945 chars): https://catalog.gatech.edu/coursesaz/econ/
OK (64,613 chars): https://catalog.gatech.edu/coursesaz/ece/
OK (596 chars): https://catalog.gatech.edu/coursesaz/free/
OK (5,634 chars): https://catalog.gatech.edu/coursesaz/gt/
OK (45,232 chars): https://catalog.gatech.edu/coursesaz/math/
OK (67,020 chars): https://catalog.gatech.edu/coursesaz/me/
OK (7,150 chars): https://catalog.gatech.edu/coursesaz/msl/
OK (27,743 chars): https://catalog.gatech.edu/coursesaz/phys/
OK (445 chars): https://catalog.gatech.edu/coursesaz/sci/
OK (597 chars): https://catalog.gatech.edu/coursesaz/ss/
OK (668 chars): https://catalog.gatec

## Saved docs to a JSON file
Persisting raw scraped data lets you re-run chunking/embedding without re-scraping.

In [ ]:
with open("catalog_docs.json","w") as f:
    json.dump(docs, f, indent=2)
print("Saved catalog_docs.json")

Saved catalog_docs.json


In [ ]:
import re

def extract_course_code(query):
    match = re.search(r"[A-Z]{2,4}\s?\d{4}", query)
    return match.group(0) if match else None

def build_search_query(query):
    course_code = extract_course_code(query)

    if course_code:
        return f"{course_code} prerequisites Georgia Tech catalog site:catalog.gatech.edu"
    else:
        return f"{query} Georgia Tech catalog"

## RAG Pipeline(chunk + embed + store)

### Chunking
Chunk_size=800 chars matches one course description.

Overlap=150 ensures prerequisite text at chunk boundaries isn't lost.

FAISS uses L2 distance over 768-dim vectors.
Saved the index locally so re-runs don't re-embed.

This is the core of any RAG pipeline , retrieval quality depends entirely on chunk design.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import FAISS

# Load saved docs
with open("catalog_docs.json") as f:
    docs = json.load(f)

# Chunk — 800 chars, 150 overlap
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n","\n",".",","," "]
)

all_chunks = []
for doc in docs:
    chunks = splitter.create_documents(
        [doc["text"]],
        metadatas=[{"url": doc["url"], "date": doc["date"]}]
    )
    all_chunks.extend(chunks)

print(f"Created {len(all_chunks)} chunks from {len(docs)} documents")


Created 963 chunks from 38 documents


## Embedding
Used GoogleGenerativeAIEmbeddings with model text-embedding-004. This produces 768-dimensional dense vectors. Each chunk becomes a point in 768D space — semantically similar chunks end up close together

In [ ]:
import google.generativeai as genai
# Listing available embedding models
print("Available embedding models:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

Available embedding models:
models/gemini-embedding-001
models/gemini-embedding-2-preview


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001" # Changed model name format
)

# Test it
test = embedding_model.embed_query("What are the prerequisites for CS 3510?")
print(f"Embedding dimension: {len(test)}")
print("Embedding model ready!")

Embedding dimension: 3072
Embedding model ready!


## Build FAISS vector store
FAISS (Facebook AI Similarity Search) builds an index using cosine similarity over the embeddings. At query time, it finds the top-k nearest vectors in milliseconds — O(n) flat index, sufficient for our dataset size.

In [ ]:
from langchain_community.vectorstores import FAISS

print("Building FAISS...")

vectorstore = FAISS.from_documents(all_chunks, embedding_model)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

print("FAISS built successfully")

Building FAISS...


GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0
Please retry in 39.687426131s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-embedding-1.0"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 100
}
, retry_delay {
  seconds: 39
}
]

As I have hit the rate limit for using the embedding model and exceeded the quota of 100 requests.
I have switched to local sentence-transformers model for vector indexing

In [ ]:
!pip install -q sentence-transformers

In [ ]:
#as the gemini embeddings exhausted switching to huggingfaceembeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("Switched to HuggingFace embeddings")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Switched to HuggingFace embeddings


In [ ]:
from langchain_community.vectorstores import FAISS

print("Building FAISS...")

vectorstore = FAISS.from_documents(all_chunks, embedding_model)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

print("FAISS built successfully")

Building FAISS...
FAISS built successfully


## Testing retrieval


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
test_results = retriever.invoke(f"CS 3101 prerequisites Georgia Tech catalog site:catalog.gatech.edu")
print(f"Retrieval test: got {len(test_results)} chunks")
print(test_results[0].page_content[:284])

Retrieval test: got 5 chunks
Georgia Tech (GT) | Georgia Tech Catalog
Catalog
2025-2026 Edition
2025-2026 Edition
Georgia Tech (GT)
GT  4900.  Special Problems.  1-21 Credit Hours.
Course of individualized instruction, which will include library, conference, and laboratory investigations.
GT 0900.  Challenge Sum


## Building RAG tool for agents
 Wraped the FAISS retriever as a LangChain Tool with a natural language description — this description is what the LLM reads to decide when to call the tool. Returning source URLs inside the result string is what enables downstream citation in the planner output.

In [ ]:
from crewai.tools import tool

@tool("search_catalog")
def catalog_tool(query: str) -> str:
    """Search the Georgia Tech course catalog for prerequisites,
    degree requirements, and academic policies.
    Input: a question about courses, prerequisites, or requirements.
    Output: relevant catalog excerpts with source URLs.
    Always use this tool before stating any prerequisite or requirement."""
    results = retriever.get_relevant_documents(build_search_query(query))
    formatted = []
    for i, r in enumerate(results):
        formatted.append(
            f"[Source {i+1}: {r.metadata['url']}]\n{r.page_content}"
        )
    return "\n\n".join(formatted)

# Quick test
_test_query = "CS 3510 prerequisites"
_test_results = retriever.get_relevant_documents(_test_query)
_test_formatted = []
for i, r in enumerate(_test_results):
    _test_formatted.append(
        f"[Source {i+1}: {r.metadata['url']}]\n{r.page_content}"
    )
test_result = "\n\n".join(_test_formatted)
print("Tool test OK:", test_result[:200])

Tool test OK: [Source 1: https://catalog.gatech.edu/coursesaz/cs/]
CS 3101.  Computer Science Ventures.  3 Credit Hours.
Students will learn how computer-science-based ventures are developed.  The course is project


## Defining the grounding system prompt
The system prompt enforces three key behaviors: citation obligation (every factual claim needs a URL), safe abstention ("I don't have that information" for out-of-scope queries), and structured output format.

This is called prompt grounding — constraining the LLM's output space to prevent hallucination. It's a core RAG design pattern.

In [ ]:
SYSTEM_PROMPT = """You are a university course planning assistant for Georgia Tech.
You ONLY use information from the provided catalog search results.

STRICT RULES:
1. Every prerequisite or requirement claim MUST include a citation: [Source: URL]
2. If the catalog does not contain the answer, respond EXACTLY:
   "I don't have that information in the provided catalog/policies."
   Then suggest: contact your academic advisor, check the department website,
   or view the Schedule of Classes.
3. Never guess. Never invent course availability, professor decisions, or waitlist info.
4. If student info is incomplete, ask 1-5 clarifying questions before planning.

OUTPUT FORMAT — always use this exact structure:
---
Answer / Plan:
[your answer or proposed course list here]

Why (requirements/prereqs satisfied):
[explain why each course is eligible — cite evidence]

Citations:
[list every URL you referenced]

Clarifying questions (if needed):
[list 1-5 questions if student profile is incomplete]

Assumptions / Not in catalog:
[anything you assumed or could not verify from the catalog]
---"""

print("System prompt defined")

System prompt defined


## Crew Agents
Multi-agent decomposition distributes cognitive load — each agent has a focused role and constrained output expectation.

allow_delegation=False enforces strict sequential pipeline, preventing the circular delegation loops that caused earlier failures.

Each agent's backstory is a mini system prompt that shapes its behavior independently of the task description.

In [ ]:
from crewai import Agent

intake_agent = Agent(
    role="Student Intake Specialist",
    goal="""
Identify missing student info ONLY if required.

If the query is about:
- course prerequisites
- catalog information
- degree requirements

DO NOT ask for student profile details.

Proceed directly using catalog knowledge.

Only ask clarifying questions if the query explicitly depends on personal student data.
""",
    backstory="You help interpret student queries efficiently.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)
retriever_agent = Agent(
    role="Catalog Retrieval Specialist",
    goal="Find exact prerequisite and requirement text from the catalog with source URLs",
    backstory="You search the catalog and return verbatim excerpts with [Source: URL] citations. Never add info not in the catalog. Do NOT delegate.",
    tools=[catalog_tool],   # now a crewai BaseTool — Pydantic validation passes
    llm=llm,
    verbose=True,
    allow_delegation=False
)

planner_agent = Agent(
    role="Academic Course Planner",
    goal="Propose an eligible course plan using only catalog-cited evidence",
    backstory="You create term plans using catalog evidence from the retriever. Every claim must have a [Source: URL]. Use the output format from the system prompt. Do NOT delegate.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

verifier_agent = Agent(
    role="Plan Auditor",
    goal="Verify every claim in the plan has a citation. Flag unsupported statements.",
    backstory="You audit plans for missing citations. If a prerequisite claim has no [Source: URL], flag it as UNSUPPORTED. Do NOT delegate.",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print("All 4 agents created — delegation disabled on all")

All 4 agents created — delegation disabled on all


## Building the crew runner function
CrewAI's sequential Process passes each task's output as context to the next task.

The 5-second sleep between crew runs prevents rate-limit errors.

Task descriptions are templated with the student query at runtime, keeping the crew reusable across all 25 eval queries.

In [ ]:
from crewai import Task, Crew, Process

def run_planning_crew(student_query: str):
    print(f"\n{'='*55}")
    print(f"QUERY: {student_query[:80]}")
    print(f"{'='*55}")

    t1 = Task(
        description=f"Student query: {student_query}\n\nIdentify any missing profile info (grades, major, term, credits). List clarifying questions if needed. If profile is complete, summarise it.",
        agent=intake_agent,
        expected_output="Student profile summary or list of clarifying questions"
    )
    t2 = Task(
        description="Using the student profile from Task 1, search the catalog for ALL relevant prerequisite chains and degree requirements. Return exact catalog text with [Source: URL] for every piece of information.",
        agent=retriever_agent,
        expected_output="Catalog excerpts with [Source: URL] citations"
    )
    t3 = Task(
        description=f"Using catalog evidence from Task 2, produce a response to: {student_query}\n\nFollow this exact format:\nAnswer / Plan:\nWhy (requirements/prereqs satisfied):\nCitations:\nClarifying questions (if needed):\nAssumptions / Not in catalog:",
        agent=planner_agent,
        expected_output="Structured answer with citations in the required format"
    )
    t4 = Task(
        description="Audit the plan from Task 3. Check every prerequisite and requirement claim has a [Source: URL]. List any UNSUPPORTED claims. Output the final verified plan.",
        agent=verifier_agent,
        expected_output="Verified final plan or flagged issues list"
    )

    crew = Crew(
        agents=[intake_agent, retriever_agent, planner_agent, verifier_agent],
        tasks=[t1, t2, t3, t4],
        process=Process.sequential,
        verbose=True
    )

    result = crew.kickoff()
    time.sleep(5)
    return result

print("run_planning_crew() is ready")

run_planning_crew() is ready


## Running Sample Interactions
The three transcript types map directly to the evaluation rubric: eligibility decision correctness, plan generation with cited justification, and abstention accuracy on out-of-scope queries. These demonstrate all 5 required capabilities simultaneously.

In [ ]:
# Transcript 1: Correct eligibility decision with citations
result1 = run_planning_crew(
    "Can I take CS 3510 (Design and Analysis of Algorithms) "
    "if I have completed CS 1332 (Data Structures) with a B?"
)
print("\n--- TRANSCRIPT 1 RESULT ---")
print(result1)



QUERY: Can I take CS 3510 (Design and Analysis of Algorithms) if I have completed CS 13
# Agent: Student Intake Specialist
## Task: Student query: Can I take CS 3510 (Design and Analysis of Algorithms) if I have completed CS 1332 (Data Structures) with a B?

Identify any missing profile info (grades, major, term, credits). List clarifying questions if needed. If profile is complete, summarise it.


# Agent: Student Intake Specialist
## Final Answer: 
The student has completed CS 1332 (Data Structures) with a B. 
To confirm eligibility for CS 3510 (Design and Analysis of Algorithms), we need to know: 
1. Are there any other prerequisites or corequisites for CS 3510 that the student needs to fulfill?
2. Does the student's major have any specific requirements that could affect their eligibility to take CS 3510?


# Agent: Catalog Retrieval Specialist
## Task: Using the student profile from Task 1, search the catalog for ALL relevant prerequisite chains and degree requirements. Return exa

ERROR:root:LiteLLM call failed: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 2348. Please try again in 33m48.672s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:root:LiteLLM call failed: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 828. Please try again in 11m55.392s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}





# Agent: Catalog Retrieval Specialist
## Thought: Thought: I need to find the prerequisites for CS 3510 to confirm eligibility for the course.
## Using tool: search_catalog
## Tool Input: 
"{\"query\": \"CS 3510 prerequisites\"}"
## Tool Output: 
[Source 1: https://catalog.gatech.edu/coursesaz/gt/]
Georgia Tech (GT) | Georgia Tech Catalog
Catalog
2025-2026 Edition
2025-2026 Edition
Georgia Tech (GT)
GT  4900.  Special Problems.  1-21 Credit Hours.
Course of individualized instruction, which will include library, conference, and laboratory investigations.
GT 0900.  Challenge Summer Intensive Residential Program: Interpersonal Development Course.  2 Credit Hours.
Challenge is a summer residential program for incoming freshman. This course provides critical skills and competencies for the interpersonal and communication development components of Challenge.
GT 1000.  First-Year Seminar.  1 Credit Hour.
Discussion of topics related to academic, social and professional success including le

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 828. Please try again in 11m55.392s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [ ]:
# Transcript 2: Full course plan with justification + citations
result2 = run_planning_crew(
    "I am a CS major at Georgia Tech. Completed: CS 1301, CS 1331, "
    "CS 1332, MATH 1551, MATH 1552. GPA 3.2. Plan 4 courses for Fall. "
    "What should I take?"
)
print("\n--- TRANSCRIPT 2 RESULT ---")
print(result2)



QUERY: I am a CS major at Georgia Tech. Completed: CS 1301, CS 1331, CS 1332, MATH 1551
# Agent: Student Intake Specialist
## Task: Student query: I am a CS major at Georgia Tech. Completed: CS 1301, CS 1331, CS 1332, MATH 1551, MATH 1552. GPA 3.2. Plan 4 courses for Fall. What should I take?

Identify any missing profile info (grades, major, term, credits). List clarifying questions if needed. If profile is complete, summarise it.


# Agent: Student Intake Specialist
## Final Answer: 
The student's profile is partially complete. The given information includes:
- Major: CS (Computer Science)
- Completed courses: 
  - CS 1301
  - CS 1331
  - CS 1332
  - MATH 1551
  - MATH 1552
- GPA: 3.2
- Planned term: Fall

However, the following information is missing or needs clarification:
- Current term or year of study
- Credit limit for the Fall term
- Specific grades for each completed course (only the overall GPA is provided)

To complete the student's profile and provide accurate advice on c

ERROR:root:LiteLLM call failed: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11968, Requested 1100. Please try again in 5.34s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

ERROR:root:LiteLLM call failed: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11948, Requested 1253. Please try again in 6.005s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}





# Agent: Academic Course Planner
## Final Answer: 
Answer / Plan: For the Fall semester, I recommend taking CS 2050: Discrete Mathematics for Computer Science, CS 2316: Data Structures and Algorithms, CS 2340: Objects and Design, and MATH 2551 or another approved math course to fulfill the degree requirements for the Bachelor of Science in Computer Science.

Why (requirements/prereqs satisfied): The recommended courses satisfy the requirements for the Bachelor of Science in Computer Science degree, including the completion of discrete mathematics, data structures and algorithms, and objects and design. Additionally, taking MATH 2551 or another approved math course will help fulfill the math requirements for the degree.

Citations: 
- The Bachelor of Science in Computer Science degree requires the following coursework: CS 1301, CS 1331, CS 1332, MATH 1551, MATH 1552, CS 2050, CS 2316, CS 2340, CS 3058, CS 3210, CS 3220, and CS 4000-level electives [Source: https://catalog.gatech.edu/d

ERROR:root:LiteLLM call failed: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11932, Requested 1387. Please try again in 6.595s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 11932, Requested 1387. Please try again in 6.595s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


## 25 Query evaluation test and output report
 Computed three automated metrics — citation_rate (URL present in response), abstention_accuracy (out-of-doc queries correctly refused using the canonical phrase), and produce a full JSON transcript for manual eligibility grading.

 The rubric is 3=correct+cited, 2=correct+weak citation, 1=wrong/hallucinated.

In [ ]:
import time

def safe_run(query, retries=3):
    for attempt in range(retries):
        try:
            return run_planning_crew(query)
        except Exception as e:
            print(f"Error: {e}")

            if "RateLimit" in str(e):
                print("Waiting due to rate limit...")
                time.sleep(5)
            else:
                time.sleep(2)

    return "FAILED AFTER RETRIES"

In [ ]:
eval_queries = [
    # Category 1: Prereq checks (10)
    {"q":"Can I take CS 3510 if I completed CS 1332 with a B?"},
    {"q":"Can I take CS 4400 if I have completed CS 1332?"},
    {"q":"What are the prerequisites for CS 1332?"},
    {"q":"Do I need CS 1331 before taking CS 1332?"},
    {"q":"Can I take CS 2340 after completing CS 1332?"},
    {"q":"What prerequisites are required for CS 3600 Artificial Intelligence?"},
    {"q":"Do I need MATH 1552 before taking higher-level CS courses?"},
    {"q":"Can I take ECE courses without completing prerequisite math courses?"},
    {"q":"Are there prerequisites for ECON courses?","cat":"prereq"},
    {"q":"Can I take PHYS courses without completing prior physics classes?"},

    # Category 2: Multi-hop chains (5)
    {"q":"What is the full prerequisite path to reach CS 4510?","cat":"chain"},
    {"q":"If I only completed CS 1301, what path should I follow to take CS 4641 Machine Learning?"},
    {"q":"What courses must I complete before taking both Algorithms and Artificial Intelligence?"},
    {"q":"Trace the prerequisite chain from CS 1301 to CS 4400."},
    {"q":"What sequence of courses is required before taking advanced CS electives?"},

    # Category 3: Program requirements (5)
    {"q":"How many total credits are required to graduate from Georgia Tech?"},
    {"q":"What are the core requirements for a Computer Science degree?"},
    {"q":"How many free electives are allowed in a degree program?"},
    {"q":"What is the minimum GPA required for graduation?"},
    {"q":"Can courses outside my major count toward graduation requirements?","cat":"program"},

    # Category 4: Out of scope / trick questions (5)
    {"q":"Is CS 3510 offered online this Fall 2025?"},
    {"q":"Who is teaching CS 4641 next semester?"},
    {"q":"What is the average grade distribution in CS 3510?"},
    {"q":"Is there a waitlist for CS 4400 this semester?"},
    {"q":"Can a professor waive prerequisites for CS courses?"},
]

In [ ]:
'''
results = []

for i, item in enumerate(eval_queries):
    print(f"\nRunning {i+1}/25: {item['q'][:55]}...")

    try:
        resp = safe_run(item["q"])   # or run_planning_crew
        resp_str = str(resp)

        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "response": resp_str,
            "has_citation": "http" in resp_str or "Source:" in resp_str,
            "abstained": "don't have that information" in resp_str.lower()
        })

    except Exception as e:
        print(f"ERROR on query {i+1}: {e}")
        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "error": str(e)
        })

    time.sleep(3)
'''


In [ ]:
# Batch 1: Prereq (first 10)
batch = eval_queries[:10]

results = []

for i, item in enumerate(batch):
    print(f"\nRunning {i+1}/10: {item['q'][:55]}...")

    try:
        resp = safe_run(item["q"])
        resp_str = str(resp)

        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "response": resp_str,
            "has_citation": "http" in resp_str or "Source:" in resp_str,
            "abstained": "don't have that information" in resp_str.lower()
        })

    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "error": str(e)
        })

    time.sleep(3)

# Save
import json
with open("eval_batch1.json", "w") as f:
    json.dump(results, f, indent=2)

print("Batch 1 saved")

In [ ]:
# Batch 2: Chains (next 5)
batch = eval_queries[10:15]

results = []

for i, item in enumerate(batch):
    print(f"\nRunning {i+1}/5: {item['q'][:55]}...")

    try:
        resp = safe_run(item["q"])
        resp_str = str(resp)

        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "response": resp_str,
            "has_citation": "http" in resp_str or "Source:" in resp_str,
            "abstained": "don't have that information" in resp_str.lower()
        })

    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "error": str(e)
        })

    time.sleep(3)

with open("eval_batch2.json", "w") as f:
    json.dump(results, f, indent=2)

print("Batch 2 saved")


Running 1/5: What is the full prerequisite path to reach CS 4510?...

QUERY: What is the full prerequisite path to reach CS 4510?
# Agent: Student Intake Specialist
## Task: Student query: What is the full prerequisite path to reach CS 4510?

Identify any missing profile info (grades, major, term, credits). List clarifying questions if needed. If profile is complete, summarise it.


ERROR:root:LiteLLM call failed: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99771, Requested 464. Please try again in 3m23.04s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}





# Agent: Student Intake Specialist
## Final Answer: 
No clarifying questions are needed, and no student profile summary is required as the query can be answered directly using catalog knowledge. The full prerequisite path to reach CS 4510 can be determined by looking at the course catalog and the prerequisites for CS 4510 and its prerequisite courses.


# Agent: Catalog Retrieval Specialist
## Task: Using the student profile from Task 1, search the catalog for ALL relevant prerequisite chains and degree requirements. Return exact catalog text with [Source: URL] for every piece of information.
Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99771, Requested 464. Please try again in 3m23.04s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing"


QUERY: What is the full prerequisite path to reach CS 4510?
# Agent: Student Intake Specialist
## Task: Student query: What is the full prerequisite path to reach CS 4510?

Identify any missing profile info (grades, major, term, credits). List clarifying questions if needed. If profile is complete, summarise it.


ERROR:root:LiteLLM call failed: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99765, Requested 400. Please try again in 2m22.56s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}



Error: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kmydz663f2yvs2d5k0ctbem6` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99765, Requested 400. Please try again in 2m22.56s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}

Waiting due to rate limit...


KeyboardInterrupt: 

In [ ]:
# Batch 3: Program (next 5)
batch = eval_queries[15:20]

results = []

for i, item in enumerate(batch):
    print(f"\nRunning {i+1}/5: {item['q'][:55]}...")

    try:
        resp = safe_run(item["q"])
        resp_str = str(resp)

        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "response": resp_str,
            "has_citation": "http" in resp_str or "Source:" in resp_str,
            "abstained": "don't have that information" in resp_str.lower()
        })

    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "error": str(e)
        })

    time.sleep(3)

with open("eval_batch3.json", "w") as f:
    json.dump(results, f, indent=2)

print("Batch 3 saved")

In [ ]:
# Batch 4: Out-of-doc (last 5)
batch = eval_queries[20:]

results = []

for i, item in enumerate(batch):
    print(f"\nRunning {i+1}/5: {item['q'][:55]}...")

    try:
        resp = safe_run(item["q"])
        resp_str = str(resp)

        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "response": resp_str,
            "has_citation": "http" in resp_str or "Source:" in resp_str,
            "abstained": "don't have that information" in resp_str.lower()
        })

    except Exception as e:
        print(f"ERROR: {e}")
        results.append({
            "id": i+1,
            "query": item["q"],
            "category": item["cat"],
            "error": str(e)
        })

    time.sleep(3)

with open("eval_batch4.json", "w") as f:
    json.dump(results, f, indent=2)

print("Batch 4 saved")

In [ ]:
import json

all_results = []

for file in [
    "eval_batch1.json",
    "eval_batch2.json",
    "eval_batch3.json",
    "eval_batch4.json"
]:
    with open(file) as f:
        all_results.extend(json.load(f))

# Metrics
total = len(all_results)
cited = sum(1 for r in all_results if r.get("has_citation"))
ood = [r for r in all_results if r.get("category")=="outofdoc"]
abstained = sum(1 for r in ood if r.get("abstained"))

print("\n" + "="*50)
print("FINAL EVALUATION REPORT")
print("="*50)
print(f"Total queries:        {total}")
print(f"Citation coverage:    {cited/total*100:.1f}% ({cited}/{total})")
print(f"Abstention accuracy:  {abstained/len(ood)*100:.1f}% ({abstained}/{len(ood)})")